In [13]:
%env NX_CUGRAPH_AUTOCONFIG=True
import networkx as nx
from itertools import combinations
from collections import defaultdict, Counter
#import igraph as ig
import pandas as pd
import sqlite3

env: NX_CUGRAPH_AUTOCONFIG=True


In [3]:
conn = sqlite3.connect("tiktok_breadth_first.db")
cursor = conn.cursor()

In [4]:
query = """
SELECT hashtag_names
FROM videos
JOIN follow_relations
ON videos.reposter_username = follow_relations.from_username
WHERE follow_relations.to_username = 'kamalahq'
"""

kamalahq_docs = cursor.execute(query).fetchall()

In [16]:
query = """
SELECT hashtag_names
FROM videos
JOIN follow_relations
ON videos.reposter_username = follow_relations.from_username
WHERE follow_relations.to_username = 'teamtrump'
"""

teamtrump_docs = cursor.execute(query).fetchall()

In [5]:
kamalahq_token = Counter()
for doc in kamalahq_docs:
    contents = doc[0].lower().split(',')
    kamalahq_token.update(contents)

In [25]:
teamtrump_token = Counter()
for doc in teamtrump_docs:
    contents = doc[0].lower().split(',')
    teamtrump_token.update(contents)

In [27]:
kamalahq_edge_weights = defaultdict(int)
for doc in kamalahq_docs:
    contents = doc[0].lower().split(',')
    tokens = sorted(set(contents)) # sorted such that edge weight pairs are indexed alphabetically
    for u, v in combinations(tokens, 2):
        kamalahq_edge_weights[(u, v)] += 1

In [28]:
teamtrump_edge_weights = defaultdict(int)
for doc in teamtrump_docs:
    contents = doc[0].lower().split(',')
    tokens = sorted(set(contents)) # sorted such that edge weight pairs are indexed alphabetically
    for u, v in combinations(tokens, 2):
        teamtrump_edge_weights[(u, v)] += 1

In [ ]:
G_k = nx.Graph()
for (u, v), weight in kamalahq_edge_weights.items():
    G_k.add_edge(u, v, weight=weight)

In [29]:
G_t = nx.Graph()
for (u, v), weight in teamtrump_edge_weights.items():
    G_t.add_edge(u, v, weight=weight)

In [10]:
nx.write_weighted_edgelist(G_k, 'kamalahq_hashtag_network.edgelist')

In [30]:
nx.write_weighted_edgelist(G_t, 'teamtrump_hashtag_network.edgelist')

In [14]:
G_k = nx.read_weighted_edgelist('kamalahq_hashtag_network.edgelist')

In [35]:
result = nx.betweenness_centrality(G_k)

KeyboardInterrupt: 

In [ ]:
sorted_dict = sorted(result.items(), key=lambda x: x[1], reverse=True)

In [ ]:
sorted_dict[:10]

[('fyp', 0.42567270632249393),
 ('foryoupage', 0.07648705487997254),
 ('viral', 0.05747155023367737),
 ('kpop', 0.054500096718386286),
 ('funny', 0.0493113542955046),
 ('foryou', 0.03941814591012687),
 ('fypシ', 0.03882751393826303),
 ('fypシ゚viral', 0.024235481050085817),
 ('sad', 0.02340932782368795),
 ('funnyvideo', 0.01796977489071221)]

In [24]:
sorted_dict[:10]

[('fyp', 0.3295462552047865),
 ('viral', 0.11921256451610235),
 ('foryou', 0.06407045710080882),
 ('foryoupage', 0.057431051554150236),
 ('2025', 0.03762651720509852),
 ('ivancornejo', 0.03652854177757405),
 ('fortnite', 0.03486471628688796),
 ('fypシ゚', 0.03364202650508145),
 ('police', 0.031591909983171886),
 ('fypシ', 0.03108109385629788)]

In [ ]:
result_t = nx.betweenness_centrality(G_t)

In [ ]:
len(kamalahq_docs) 

1298363